<a href="https://colab.research.google.com/github/vikashkumar-raj/AI-Candlestick-Prediction/blob/main/Gold_AI_Project2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==============================================================================
# 🥇 PHASE 1: DATA INGESTION & STRUCTURAL SCHEMA VALIDATION (SEMICOLON RESOLVED)
# Folder Mapping: src/data/loader.py
# ==============================================================================
import os
import pandas as pd
import numpy as np
from google.colab import drive

print("==========================================================")
print("📥 PHASE 1: PARSING Semicolon DELIMITER & LOADING DATA")
print("==========================================================")

def execute_phase_1_with_drive():
    """
    Mounts drive, reads semicolon separated raw logs, unpacks features cleanly,
    validates the structural framework, and chronologically tracks entries.
    """
    # 1. Secure Drive connection checkpoint
    print("[1/4] Securing active Google Drive context...")
    drive.mount('/content/drive', force_remount=True)

    # 2. Hard-coded prioritized path matrix from previous step
    possible_paths = [
        '/content/drive/MyDrive/Gold-AI-Trading-System/datasets/XAU_1h_data.csv',
        '/content/drive/MyDrive/Gold-AI-Trading-System/datasets/raw/XAU_1h.csv',
        '/content/drive/MyDrive/XAU_1h_data.csv',
        'XAU_1h_data.csv'
    ]

    target_path = None
    for path in possible_paths:
        if os.path.exists(path):
            target_path = path
            break

    if target_path is None:
        raise FileNotFoundError("🔴 Critical Error: File could not be located in the allocated directories.")

    print(f"[2/4] Parsing file source with Semicolon rules: '{target_path}'")

    # 3. Reading with semicolon specifier to parse distinct features cleanly
    raw_df = pd.read_csv(target_path, sep=';')

    # Correcting dynamic header text artifact if any mapping error happens
    raw_df.columns = [col.split(';')[0].strip() for col in raw_df.columns]

    # 4. Mandatory column checking baseline validation
    mandatory_schema = ['Date', 'Open', 'High', 'Low', 'Close', 'Volume']
    missing_columns = [col for col in mandatory_schema if col not in raw_df.columns]

    if missing_columns:
        raise KeyError(f"🔴 Schema Configuration Failed! Missing features: {missing_columns}")
    print("   ✓ Core OHLCV + Volume features split successfully.")

    # 5. Advanced formatting logic to handle timestamp variations (YYYY.MM.DD HH:MM)
    print("[3/4] Unpacking and standardizing time signatures...")
    raw_df['Date'] = raw_df['Date'].astype(str).str.replace('.', '-', regex=False)
    raw_df['Date'] = pd.to_datetime(raw_df['Date'], errors='raise')

    # Ensure numerical features are strictly typed to float32/int32 vectors
    for num_col in ['Open', 'High', 'Low', 'Close', 'Volume']:
        raw_df[num_col] = pd.to_numeric(raw_df[num_col], errors='coerce')

    print("[4/4] Locking strict chronological historical array execution sequence...")
    processed_df = raw_df.sort_values('Date').reset_index(drop=True)

    # 6. Metadata profile print summary
    total_bars = len(processed_df)
    print("\n==========================================================")
    print("🎯 PHASE 1: REPOSITORY DATA INGESTION PROFILE REPORT")
    print("==========================================================")
    print(f"  • Resolved Path           : {target_path}")
    print(f"  • Total Time Bars Ingested: {total_bars}")
    print(f"  • Timeline Tracking From  : {processed_df['Date'].min()}")
    print(f"  • Timeline Tracking To    : {processed_df['Date'].max()}")
    print("==========================================================")
    print("✅ Phase 1 data blocks completely mapped and locked.")

    return processed_df

# Run Phase 1 execution
df_1h = execute_phase_1_with_drive()

📥 PHASE 1: PARSING Semicolon DELIMITER & LOADING DATA
[1/4] Securing active Google Drive context...
Mounted at /content/drive
[2/4] Parsing file source with Semicolon rules: '/content/drive/MyDrive/Gold-AI-Trading-System/datasets/XAU_1h_data.csv'
   ✓ Core OHLCV + Volume features split successfully.
[3/4] Unpacking and standardizing time signatures...
[4/4] Locking strict chronological historical array execution sequence...

🎯 PHASE 1: REPOSITORY DATA INGESTION PROFILE REPORT
  • Resolved Path           : /content/drive/MyDrive/Gold-AI-Trading-System/datasets/XAU_1h_data.csv
  • Total Time Bars Ingested: 125206
  • Timeline Tracking From  : 2004-06-11 07:00:00
  • Timeline Tracking To    : 2026-01-30 23:00:00
✅ Phase 1 data blocks completely mapped and locked.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# ==============================================================================
# 🥇 PHASE 2: EXPLORATORY DATA ANALYSIS & INTEGRITY VERIFICATION (1H TIMEFRAME)
# Folder Mapping: src/data/validator.py & src/data/cleaner.py
# ==============================================================================
import numpy as np
import pandas as pd

print("==========================================================")
print("🔍 PHASE 2: INITIALIZING CORE DATA INTEGRITY DIAGNOSTICS")
print("==========================================================")

def execute_phase_2_validation(df):
    """
    Runs multi-point integrity checks for numeric anomalies, zero valuation
    boundaries, time gaps, and duplicate rows across the loaded asset stream.
    """
    # Create an isolated local copy to prevent setting-with-copy warnings
    working_df = df.copy()

    print("[1/3] Scanning timeline dimensions for temporal anomalies...")
    # 1. Deduplicate timeline sequences based on sorted datetime index
    duplicate_count = working_df['Date'].duplicated().sum()
    if duplicate_count > 0:
        print(f"   ⚠️ Warning: Detected {duplicate_count} overlapping duplicate rows. Clearing duplicates...")
        working_df = working_df.drop_duplicates(subset=['Date']).reset_index(drop=True)
    else:
        print("   ✓ Integrity Pass: Timeline sequence has zero tracking overlaps.")

    print("[2/3] Auditing matrix cell arrays for NaN and Infinity values...")
    # 2. Extract numeric feature targets for statistical sanity testing
    numeric_columns = ['Open', 'High', 'Low', 'Close', 'Volume']

    # Evaluate data leakage or broken metrics array cells
    nan_cells = working_df[numeric_columns].isna().sum().to_dict()
    inf_cells = np.isinf(working_df[numeric_columns]).sum().to_dict()

    print(f"   • Null/NaN Cell Map : {nan_cells}")
    print(f"   • Infinite Bound Map: {inf_cells}")

    # 3. Structural Boundary Verification: Ensure absolute prices are positive real numbers
    illegal_bounds = (working_df[['Open', 'High', 'Low', 'Close']] <= 0).sum().sum()
    if illegal_bounds > 0:
        raise ValueError("🔴 Data Invalidation: Negative or absolute zero market evaluation discovered in baseline matrix rows.")

    print("[3/3] Engineering underlying descriptive metrics report...")
    # 4. Generate high-precision asset baseline statistics
    ohlc_summary = working_df[numeric_columns].describe().T

    print("\n==========================================================")
    print("📊 PHASE 2: DATA INTEGRITY & PHYSICAL RUNTIME LOG")
    print("==========================================================")
    print(f"  • Post-Sanitization Records : {len(working_df)}")
    print(f"  • Asset Global Floor (Low)  : {working_df['Low'].min():.2f}")
    print(f"  • Asset Global Peak (High)  : {working_df['High'].max():.2f}")
    print(f"  • Mean Volume Distribution  : {working_df['Volume'].mean():.2f}")
    print("==========================================================")
    print("✅ Phase 2 diagnostic evaluations complete. Dataframe purity locked.")

    return working_df

# Trigger Phase 2 Execution using the df_1h vector array variable from Phase 1
df_1h = execute_phase_2_validation(df_1h)

🔍 PHASE 2: INITIALIZING CORE DATA INTEGRITY DIAGNOSTICS
[1/3] Scanning timeline dimensions for temporal anomalies...
   ✓ Integrity Pass: Timeline sequence has zero tracking overlaps.
[2/3] Auditing matrix cell arrays for NaN and Infinity values...
   • Null/NaN Cell Map : {'Open': 0, 'High': 0, 'Low': 0, 'Close': 0, 'Volume': 0}
   • Infinite Bound Map: {'Open': 0, 'High': 0, 'Low': 0, 'Close': 0, 'Volume': 0}
[3/3] Engineering underlying descriptive metrics report...

📊 PHASE 2: DATA INTEGRITY & PHYSICAL RUNTIME LOG
  • Post-Sanitization Records : 125206
  • Asset Global Floor (Low)  : 381.10
  • Asset Global Peak (High)  : 5597.60
  • Mean Volume Distribution  : 3900.01
✅ Phase 2 diagnostic evaluations complete. Dataframe purity locked.


In [ ]:
# ==============================================================================
# 🥇 PHASE 3: CANDLESTICK DETECTION ENGINE (1H TIMEFRAME - SYNTAX FIXED)
# Folder Mapping: src/candlestick/detector.py
# ==============================================================================
import numpy as np
import pandas as pd

print("==========================================================")
print("🕯️ PHASE 3: INITIALIZING ARCHITECTURAL CANDLESTICK DETECTION")
print("==========================================================")

def execute_phase_3_candlestick_engine(df):
    """
    Vectorized structural mathematical engine that scans and maps 22 distinct
    candlestick formations precisely without shifting windows to preserve temporal purity.
    """
    working_df = df.copy()

    # --- HELPER BASE VALUATIONS (Vectorized Operations) ---
    O = working_df['Open'].to_numpy()
    H = working_df['High'].to_numpy()
    L = working_df['Low'].to_numpy()
    C = working_df['Close'].to_numpy()

    # Core body and shadow boundaries
    body = np.abs(C - O)
    candle_range = H - L
    # Prevent divide-by-zero on completely flat bars
    candle_range = np.where(candle_range == 0, 1e-5, candle_range)

    body_max = np.maximum(O, C)
    body_min = np.minimum(O, C)

    upper_shadow = H - body_max
    lower_shadow = body_min - L

    is_bullish = C > O
    is_bearish = C < O

    # Average body calculation for standard comparison metrics (rolling window context)
    avg_body = working_df['Close'].diff().abs().rolling(window=10, min_periods=1).mean().to_numpy()
    avg_body = np.nan_to_num(avg_body, nan=1e-5)

    # --------------------------------------------------------------------------
    # VECTORIZED STRUCTURAL PATTERN DETECTION LOGIC (22 PATTERNS)
    # --------------------------------------------------------------------------

    # 1. Doji
    working_df['Pattern_Doji'] = (body <= (candle_range * 0.1)).astype(int)

    # 2. Marubozu
    working_df['Pattern_Marubozu'] = ((body >= (candle_range * 0.9)) & (upper_shadow <= (candle_range * 0.05)) & (lower_shadow <= (candle_range * 0.05))).astype(int)

    # 3. Hammer
    working_df['Pattern_Hammer'] = ((lower_shadow >= (body * 2)) & (upper_shadow <= (candle_range * 0.1)) & (body <= (candle_range * 0.35))).astype(int)

    # 4. Inverted Hammer
    working_df['Pattern_Inverted_Hammer'] = ((upper_shadow >= (body * 2)) & (lower_shadow <= (candle_range * 0.1)) & (body <= (candle_range * 0.35))).astype(int)

    # 5. Hanging Man
    working_df['Pattern_Hanging_Man'] = (is_bearish & (lower_shadow >= (body * 2)) & (upper_shadow <= (candle_range * 0.1))).astype(int)

    # 6. Shooting Star
    working_df['Pattern_Shooting_Star'] = (is_bearish & (upper_shadow >= (body * 2)) & (lower_shadow <= (candle_range * 0.1))).astype(int)

    # 7. Spinning Top
    working_df['Pattern_Spinning_Top'] = ((body <= (candle_range * 0.3)) & (upper_shadow >= (body * 0.5)) & (lower_shadow >= (body * 0.5)) & (working_df['Pattern_Doji'] == 0)).astype(int)

    # Multi-bar shifts for sequential structural verification
    p_bull_eng = np.zeros(len(working_df), dtype=int)
    p_bear_eng = np.zeros(len(working_df), dtype=int)
    p_morning = np.zeros(len(working_df), dtype=int)
    p_evening = np.zeros(len(working_df), dtype=int)
    p_harami = np.zeros(len(working_df), dtype=int)
    p_piercing = np.zeros(len(working_df), dtype=int)
    p_dark_cloud = np.zeros(len(working_df), dtype=int)
    p_three_soldiers = np.zeros(len(working_df), dtype=int)
    p_three_crows = np.zeros(len(working_df), dtype=int)
    p_tweezer_top = np.zeros(len(working_df), dtype=int)
    p_tweezer_bottom = np.zeros(len(working_df), dtype=int)
    p_inside_up = np.zeros(len(working_df), dtype=int)
    p_inside_down = np.zeros(len(working_df), dtype=int)
    p_outside_up = np.zeros(len(working_df), dtype=int)
    p_outside_down = np.zeros(len(working_df), dtype=int)

    # Iterative loop execution safely optimized
    for i in range(2, len(working_df)):
        # 8. Bullish Engulfing
        if is_bearish[i-1] and is_bullish[i] and C[i] >= O[i-1] and O[i] <= C[i-1]:
            p_bull_eng[i] = 1

        # 9. Bearish Engulfing
        if is_bullish[i-1] and is_bearish[i] and C[i] <= O[i-1] and O[i] >= C[i-1]:
            p_bear_eng[i] = 1

        # 10. Harami
        if body[i-1] > avg_body[i-1] and body[i] < body[i-1] and body_max[i] <= body_max[i-1] and body_min[i] >= body_min[i-1]:
            p_harami[i] = 1

        # 11. Piercing Line
        if is_bearish[i-1] and is_bullish[i] and O[i] < L[i-1] and C[i] > (O[i-1] + C[i-1])/2 and C[i] < O[i-1]:
            p_piercing[i] = 1

        # 12. Dark Cloud Cover
        if is_bullish[i-1] and is_bearish[i] and O[i] > H[i-1] and C[i] < (O[i-1] + C[i-1])/2 and C[i] > O[i-1]:
            p_dark_cloud[i] = 1

        # 13. Morning Star
        if is_bearish[i-2] and body[i-1] < (avg_body[i-2] * 0.5) and is_bullish[i] and C[i] > (O[i-2] + C[i-2])/2:
            p_morning[i] = 1

        # 14. Evening Star
        if is_bullish[i-2] and body[i-1] < (avg_body[i-2] * 0.5) and is_bearish[i] and C[i] < (O[i-2] + C[i-2])/2:
            p_evening[i] = 1

        # 15. Three White Soldiers
        if is_bullish[i-2] and is_bullish[i-1] and is_bullish[i] and C[i] > C[i-1] and C[i-1] > C[i-2]:
            p_three_soldiers[i] = 1

        # 16. Three Black Crows
        if is_bearish[i-2] and is_bearish[i-1] and is_bearish[i] and C[i] < C[i-1] and C[i-1] < C[i-2]:
            p_three_crows[i] = 1

        # 17. Tweezer Top
        if np.abs(H[i] - H[i-1]) <= (candle_range[i] * 0.05) and is_bullish[i-1] and is_bearish[i]:
            p_tweezer_top[i] = 1

        # 18. Tweezer Bottom
        if np.abs(L[i] - L[i-1]) <= (candle_range[i] * 0.05) and is_bearish[i-1] and is_bullish[i]:
            p_tweezer_bottom[i] = 1

        # 19. Three Inside Up
        harami_up_check = (body[i-1] < body[i-2]) and (body_max[i-1] <= body_max[i-2])
        if is_bearish[i-2] and harami_up_check and is_bullish[i] and C[i] > C[i-1]:
            p_inside_up[i] = 1

        # 20. Three Inside Down
        harami_down_check = (body[i-1] < body[i-2]) and (body_min[i-1] >= body_min[i-2])
        if is_bullish[i-2] and harami_down_check and is_bearish[i] and C[i] < C[i-1]:
            p_inside_down[i] = 1

        # 21. Three Outside Up
        if is_bearish[i-2] and is_bullish[i-1] and C[i-1] >= O[i-2] and is_bullish[i] and C[i] > C[i-1]:
            p_outside_up[i] = 1

        # 22. Three Outside Down
        if is_bullish[i-2] and is_bearish[i-1] and C[i-1] <= O[i-2] and is_bearish[i] and C[i] < C[i-1]:
            p_outside_down[i] = 1

    # Assign mapped patterns safely to features space
    working_df['Pattern_Bullish_Engulfing'] = p_bull_eng
    working_df['Pattern_Bearish_Engulfing'] = p_bear_eng
    working_df['Pattern_Harami'] = p_harami
    working_df['Pattern_Piercing'] = p_piercing
    working_df['Pattern_Dark_Cloud'] = p_dark_cloud
    working_df['Pattern_Morning_Star'] = p_morning
    working_df['Pattern_Evening_Star'] = p_evening
    working_df['Pattern_Three_White_Soldiers'] = p_three_soldiers
    working_df['Pattern_Three_Black_Crows'] = p_three_crows
    working_df['Pattern_Tweezer_Top'] = p_tweezer_top
    working_df['Pattern_Tweezer_Bottom'] = p_tweezer_bottom
    working_df['Pattern_Three_Inside_Up'] = p_inside_up
    working_df['Pattern_Three_Inside_Down'] = p_inside_down
    working_df['Pattern_Three_Outside_Up'] = p_outside_up
    working_df['Pattern_Three_Outside_Down'] = p_outside_down

    pattern_cols = [col for col in working_df.columns if col.startswith('Pattern_')]
    total_detections = working_df[pattern_cols].sum().sum()

    print("\n==========================================================")
    print("📊 PHASE 3: CANDLESTICK INTELLIGENCE SIGNAL DETECTOR REPORT")
    print("==========================================================")
    print(f"  • Candlestick Vector Space Formed: {len(pattern_cols)} Patterns Active")
    print(f"  • Global Formations Flagged     : {total_detections} Occurrences")
    print("==========================================================")
    print("✅ Phase 3 candlestick feature generation complete without errors.")

    return working_df

# Trigger Phase 3 Execution safely
df_1h = execute_phase_3_candlestick_engine(df_1h)

🕯️ PHASE 3: INITIALIZING ARCHITECTURAL CANDLESTICK DETECTION

📊 PHASE 3: CANDLESTICK INTELLIGENCE SIGNAL DETECTOR REPORT
  • Candlestick Vector Space Formed: 22 Patterns Active
  • Global Formations Flagged     : 174471 Occurrences
✅ Phase 3 candlestick feature generation complete without errors.


In [ ]:
# ==============================================================================
# 🥇 PHASE 4: TECHNICAL INDICATORS ENGINE (1H TIMEFRAME)
# Folder Mapping: src/indicators/indicator_engine.py & individual component files
# ==============================================================================
import numpy as np
import pandas as pd

print("==========================================================")
print("⚙️ PHASE 4: COMPUTING STRUCTURAL TECHNICAL INDICATORS")
print("==========================================================")

def execute_phase_4_indicator_engine(df):
    """
    Vectorized structural indicator calculator implementing 9 specified trading
    metrics using native pandas and numpy window matrices.
    """
    working_df = df.copy()

    # Pre-extract values for fast matrix calculations
    O = working_df['Open'].to_numpy()
    H = working_df['High'].to_numpy()
    L = working_df['Low'].to_numpy()
    C = working_df['Close'].to_numpy()
    V = working_df['Volume'].to_numpy()

    # --------------------------------------------------------------------------
    # 1. Exponential Moving Averages (EMA20, EMA50, EMA200)
    # --------------------------------------------------------------------------
    print("   • Calculating Trend Lifelines: EMA20, EMA50, EMA200...")
    working_df['Indicator_EMA20'] = working_df['Close'].ewm(span=20, adjust=False).mean()
    working_df['Indicator_EMA50'] = working_df['Close'].ewm(span=50, adjust=False).mean()
    working_df['Indicator_EMA200'] = working_df['Close'].ewm(span=200, adjust=False).mean()

    # --------------------------------------------------------------------------
    # 2. Relative Strength Index (RSI - 14 Period Wilders Exponential Smoothing)
    # --------------------------------------------------------------------------
    print("   • Calculating Momentum Spaces: RSI14...")
    delta = working_df['Close'].diff().to_numpy()
    gain = np.where(delta > 0, delta, 0.0)
    loss = np.where(delta < 0, -delta, 0.0)

    avg_gain = pd.Series(gain).ewm(com=13, adjust=False).mean().to_numpy()
    avg_loss = pd.Series(loss).ewm(com=13, adjust=False).mean().to_numpy()
    # Avoid zero division inside oscillator limits
    avg_loss = np.where(avg_loss == 0, 1e-5, avg_loss)

    rs = avg_gain / avg_loss
    working_df['Indicator_RSI'] = 100 - (100 / (1 + rs))

    # --------------------------------------------------------------------------
    # 3. Moving Average Convergence Divergence (MACD 12, 26, 9)
    # --------------------------------------------------------------------------
    print("   • Calculating Volatility Waves: MACD Line, Signal Line, Hist...")
    ema12 = working_df['Close'].ewm(span=12, adjust=False).mean()
    ema26 = working_df['Close'].ewm(span=26, adjust=False).mean()

    working_df['Indicator_MACD'] = ema12 - ema26
    working_df['Indicator_MACD_Signal'] = working_df['Indicator_MACD'].ewm(span=9, adjust=False).mean()
    working_df['Indicator_MACD_Hist'] = working_df['Indicator_MACD'] - working_df['Indicator_MACD_Signal']

    # --------------------------------------------------------------------------
    # 4. Average True Range (ATR - 14 Period True Range Bound)
    # --------------------------------------------------------------------------
    print("   • Calculating Risk Limits: ATR14...")
    prev_close = working_df['Close'].shift(1).to_numpy()
    prev_close[0] = C[0] # Handle boundary zero artifact

    tr1 = H - L
    tr2 = np.abs(H - prev_close)
    tr3 = np.abs(L - prev_close)

    true_range = np.maximum(tr1, np.maximum(tr2, tr3))
    working_df['Indicator_ATR'] = pd.Series(true_range).ewm(span=14, adjust=False).mean().to_numpy()

    # --------------------------------------------------------------------------
    # 5. Average Directional Index (ADX - 14 Period Strength Vector)
    # --------------------------------------------------------------------------
    print("   • Calculating Directional Forces: ADX14...")
    up_move = working_df['High'].diff().to_numpy()
    down_move = working_df['Low'].diff().to_numpy()
    # Mask negative variations
    up_move[0] = down_move[0] = 0.0

    plus_dm = np.where((up_move > down_move) & (up_move > 0), up_move, 0.0)
    minus_dm = np.where((down_move > up_move) & (down_move > 0), down_move, 0.0)

    atr_smooth = working_df['Indicator_ATR'].to_numpy()
    atr_smooth = np.where(atr_smooth == 0, 1e-5, atr_smooth) # Boundary safe rule

    plus_di = 100 * (pd.Series(plus_dm).ewm(span=14, adjust=False).mean().to_numpy() / atr_smooth)
    minus_di = 100 * (pd.Series(minus_dm).ewm(span=14, adjust=False).mean().to_numpy() / atr_smooth)

    di_sum = plus_di + minus_di
    di_sum = np.where(di_sum == 0, 1e-5, di_sum)
    dx = 100 * (np.abs(plus_di - minus_di) / di_sum)
    working_df['Indicator_ADX'] = pd.Series(dx).ewm(span=14, adjust=False).mean().to_numpy()

    # --------------------------------------------------------------------------
    # 6. Volume Weighted Average Price (VWAP - Cumulative Session Variant)
    # --------------------------------------------------------------------------
    print("   • Calculating Institutional Anchors: VWAP...")
    typical_price = (H + L + C) / 3.0
    cum_tp_v = (typical_price * V).cumsum()
    cum_v = V.cumsum()
    cum_v = np.where(cum_v == 0, 1e-5, cum_v)
    working_df['Indicator_VWAP'] = cum_tp_v / cum_v

    # --------------------------------------------------------------------------
    # 7. On-Balance Volume (OBV - Momentum Flow Strategy Matrix)
    # --------------------------------------------------------------------------
    print("   • Calculating Order Flows: OBV Accumulation...")
    obv = np.zeros(len(working_df))
    for i in range(1, len(working_df)):
        if C[i] > C[i-1]:
            obv[i] = obv[i-1] + V[i]
        elif C[i] < C[i-1]:
            obv[i] = obv[i-1] - V[i]
        else:
            obv[i] = obv[i-1]
    working_df['Indicator_OBV'] = obv

    # Handle nan fills across historical back-step boundaries
    indicator_cols = [col for col in working_df.columns if col.startswith('Indicator_')]
    working_df[indicator_cols] = working_df[indicator_cols].ffill().bfill().fillna(0.0)

    print("\n==========================================================")
    print("📊 PHASE 4: TECHNICAL INDICATORS EXTRACTION MATRIX REPORT")
    print("==========================================================")
    print(f"  • Mathematical Oscillators Locked: {len(indicator_cols)} Indicator Streams")
    print(f"  • Structural Array Completeness  : 100.00% Clean Numerical Space")
    print("==========================================================")
    print("✅ Phase 4 Technical Engineering layer closed successfully.")

    return working_df

# Trigger Phase 4 Execution safely
df_1h = execute_phase_4_indicator_engine(df_1h)

⚙️ PHASE 4: COMPUTING STRUCTURAL TECHNICAL INDICATORS
   • Calculating Trend Lifelines: EMA20, EMA50, EMA200...
   • Calculating Momentum Spaces: RSI14...
   • Calculating Volatility Waves: MACD Line, Signal Line, Hist...
   • Calculating Risk Limits: ATR14...
   • Calculating Directional Forces: ADX14...
   • Calculating Institutional Anchors: VWAP...
   • Calculating Order Flows: OBV Accumulation...

📊 PHASE 4: TECHNICAL INDICATORS EXTRACTION MATRIX REPORT
  • Mathematical Oscillators Locked: 11 Indicator Streams
  • Structural Array Completeness  : 100.00% Clean Numerical Space
✅ Phase 4 Technical Engineering layer closed successfully.


In [ ]:
# ==============================================================================
# 🥇 PHASE 5: HISTORICAL STATISTICS ENGINE (1H TIMEFRAME)
# Folder Mapping: src/statistics/statistics_engine.py & structural files
# ==============================================================================
import numpy as np
import pandas as pd

print("==========================================================")
print("📊 PHASE 5: EVALUATING HISTORICAL PATTERN INTELLIGENCE STATS")
print("==========================================================")

def execute_phase_5_statistics_engine(df):
    """
    Evaluates the historical efficacy of each detected pattern based on a forward
    4-candle window using a strict 1:2 Risk-Reward boundary condition.
    """
    working_df = df.copy()

    # 1. Isolate pattern columns and price tracking targets
    pattern_cols = [col for col in working_df.columns if col.startswith('Pattern_')]

    # Pre-extract data vectors to avoid inner loop Pandas overhead
    O = working_df['Open'].to_numpy()
    H = working_df['High'].to_numpy()
    L = working_df['Low'].to_numpy()
    C = working_df['Close'].to_numpy()
    n_records = len(working_df)

    # 2. Pre-calculate outcomes for every bar to evaluate signals efficiently
    # 1 = Bullish win, -1 = Bearish win, 0 = Timeout/Loss
    forward_bullish_win = np.zeros(n_records, dtype=int)
    forward_bearish_win = np.zeros(n_records, dtype=int)

    print("[1/2] Computing forward 4-candle structural boundaries...")
    for i in range(n_records - 4):
        entry = O[i + 1] # Market entry at next open price boundary

        # Bullish Boundaries (Using previous bar Low as SL)
        sl_bull = L[i]
        risk_bull = entry - sl_bull
        if risk_bull <= 0: risk_bull = 1e-5
        tp_bull = entry + (2.0 * risk_bull)

        # Bearish Boundaries (Using previous bar High as SL)
        sl_bear = H[i]
        risk_bear = sl_bear - entry
        if risk_bear <= 0: risk_bear = 1e-5
        tp_bear = entry - (2.0 * risk_bear)

        # Scan next 4 candles for resolution
        for step in range(1, 5):
            idx = i + step
            # Check Bullish path
            if L[idx] <= sl_bull:
                break # Stopped out
            if H[idx] >= tp_bull:
                forward_bullish_win[i] = 1
                break

        for step in range(1, 5):
            idx = i + step
            # Check Bearish path
            if H[idx] >= sl_bear:
                break # Stopped out
            if L[idx] <= tp_bear:
                forward_bearish_win[i] = 1
                break

    # 3. Compile Performance Metrics Dictionary for XAI Support Layers
    print("[2/2] Generating comprehensive profile dictionary for active schemas...")
    stats_profile = {}

    for col in pattern_cols:
        indices = working_df[working_df[col] == 1].index.to_numpy()
        occurrences = len(indices)

        if occurrences == 0:
            stats_profile[col] = {"Occurrence": 0, "Success_Rate": 0.0, "Expected_Value": 0.0, "Reliability": "Low"}
            continue

        # Determine performance profile based on standard naming structures
        is_bull_pattern = any(x in col.lower() for x in ['bullish', 'hammer', 'morning', 'soldiers', 'inside_up', 'outside_up', 'tweezer_bottom'])

        wins = 0
        for idx in indices:
            if is_bull_pattern and forward_bullish_win[idx] == 1:
                wins += 1
            elif not is_bull_pattern and forward_bearish_win[idx] == 1:
                wins += 1

        success_rate = (wins / occurrences) * 100.0

        # Expected value calculation framework: (Win% * 2.0 RR) - (Loss% * 1.0 Risk)
        ev = ((success_rate / 100.0) * 2.0) - ((1.0 - (success_rate / 100.0)) * 1.0)

        reliability = "High" if success_rate >= 55.0 and occurrences >= 30 else "Moderate" if success_rate >= 45.0 else "Low"

        stats_profile[col] = {
            "Occurrence": occurrences,
            "Success_Rate": round(success_rate, 2),
            "Expected_Value": round(ev, 3),
            "Reliability": reliability
        }

    # Save dictionary metadata back to global script context for the downstream XAI layer
    working_df.attrs['pattern_stats_profile'] = stats_profile

    # Quick printout summary of top performing patterns detected
    print("\n==========================================================")
    print("🎯 PHASE 5: HISTORICAL INTELLIGENCE STATISTICAL HIGHLIGHTS")
    print("==========================================================")
    sorted_patterns = sorted(stats_profile.items(), key=lambda x: x[1]['Success_Rate'], reverse=True)[:3]
    for name, metrics in sorted_patterns:
        print(f"  • {name:<30} | Occur: {metrics['Occurrence']:<4} | WinRate: {metrics['Success_Rate']}% | EV: {metrics['Expected_Value']}")
    print("==========================================================")
    print("✅ Phase 5 historical statistics locked into dataframe properties.")

    return working_df

# Trigger Phase 5 Execution safely
df_1h = execute_phase_5_statistics_engine(df_1h)

📊 PHASE 5: EVALUATING HISTORICAL PATTERN INTELLIGENCE STATS
[1/2] Computing forward 4-candle structural boundaries...
[2/2] Generating comprehensive profile dictionary for active schemas...

🎯 PHASE 5: HISTORICAL INTELLIGENCE STATISTICAL HIGHLIGHTS
  • Pattern_Spinning_Top           | Occur: 21509 | WinRate: 25.72% | EV: -0.228
  • Pattern_Doji                   | Occur: 14261 | WinRate: 22.99% | EV: -0.31
  • Pattern_Harami                 | Occur: 12471 | WinRate: 22.91% | EV: -0.313
✅ Phase 5 historical statistics locked into dataframe properties.


In [ ]:
# ==============================================================================
# 🥇 PHASE 6 UPGRADE: MULTI-TASK FEATURE SELECTION & TARGET LABELLING
# Folder Mapping: src/features/feature_engineering.py
# ==============================================================================
import numpy as np
import pandas as pd

print("==========================================================")
print("🔧 PHASE 6: GENERATING MULTI-TASK TARGET LABELS")
print("==========================================================")

def execute_phase_6_multi_task(df):
    working_df = df.copy()

    # 1. Base transformations
    working_df['Feature_Log_Volume'] = np.log1p(np.maximum(working_df['Volume'].to_numpy(), 0.0))
    obv_array = working_df['Indicator_OBV'].to_numpy()
    working_df['Feature_Log_OBV'] = np.sign(obv_array) * np.log1p(np.abs(obv_array))

    # 2. Continuous Metric Scaling Baseline
    continuous_features = [
        'Open', 'High', 'Low', 'Close',
        'Indicator_EMA20', 'Indicator_EMA50', 'Indicator_EMA200',
        'Indicator_RSI', 'Indicator_MACD', 'Indicator_MACD_Signal', 'Indicator_MACD_Hist',
        'Indicator_ATR', 'Indicator_ADX', 'Indicator_VWAP', 'Feature_Log_Volume', 'Feature_Log_OBV'
    ]
    for col in continuous_features:
        col_mean = working_df[col].mean()
        col_std = working_df[col].std()
        if col_std == 0: col_std = 1e-5
        working_df[f'Scaled_{col}'] = (working_df[col] - col_mean) / col_std

    # --------------------------------------------------------------------------
    # DUAL TARGET LOGIC FOR CLASSIFICATION & REGRESSION
    # --------------------------------------------------------------------------
    next_close = working_df['Close'].shift(-1).to_numpy()
    current_close = working_df['Close'].to_numpy()

    # A. Classification Target: Directional Binary Vector (Up=1, Down=0)
    working_df['Target_Class'] = (next_close > current_close).astype(int)

    # B. Regression Target: Continuous Magnitude Return Vector (Price Difference)
    working_df['Target_Reg'] = next_close - current_close

    # Clean boundary constraints
    working_df = working_df.iloc[:-1].reset_index(drop=True)
    final_features = [col for col in working_df.columns if col.startswith('Scaled_') or col.startswith('Pattern_')]

    print("✅ Dual Targets Created: 'Target_Class' & 'Target_Reg' successfully mapped.")
    return working_df, final_features

df_1h, feature_columns_list = execute_phase_6_multi_task(df_1h)

🔧 PHASE 6: GENERATING MULTI-TASK TARGET LABELS
✅ Dual Targets Created: 'Target_Class' & 'Target_Reg' successfully mapped.


In [ ]:
# ==============================================================================
# 🥇 PHASE 7: PURE STANDALONE SEQUENCE GENERATOR (NO MULTI-TASK DICTIONARY LEAK)
# Folder Mapping: src/data/sequence_generator.py
# ==============================================================================
import numpy as np
import tensorflow as tf

print("==========================================================")
print("⚙️ PHASE 7: BUILDING CLEAN STANDALONE SEQUENCE GENERATOR")
print("==========================================================")

class StandaloneSequenceStream(tf.keras.utils.Sequence):
    """
    Production-grade sequence generator optimized for standalone models.
    Yields 3D feature matrices and a single flat binary classification array.
    """
    def __init__(self, feature_matrix, label_vector, sequence_length, batch_size):
        self.feature_matrix = feature_matrix.astype(np.float32)
        self.label_vector = label_vector.astype(np.int32)
        self.sequence_length = sequence_length
        self.batch_size = batch_size
        # Valid starting index boundaries to ensure lookback window fits
        self.valid_indices = np.arange(sequence_length, len(feature_matrix))

    def __len__(self):
        return int(np.ceil(len(self.valid_indices) / self.batch_size))

    def __getitem__(self, index):
        # Isolate the index boundaries for the current batch
        batch_slice = self.valid_indices[index * self.batch_size : (index + 1) * self.batch_size]
        actual_batch_size = len(batch_slice)

        # Pre-allocate 3D Input Tensor shapes: [Batch_Size, Sequence_Length, Features]
        X_batch = np.empty((actual_batch_size, self.sequence_length, self.feature_matrix.shape[1]), dtype=np.float32)
        y_batch = np.empty((actual_batch_size,), dtype=np.int32)

        for idx, end_point in enumerate(batch_slice):
            # Window slicing from (end_point - 30) up to end_point
            X_batch[idx] = self.feature_matrix[end_point - self.sequence_length : end_point]
            y_batch[idx] = self.label_vector[end_point]

        # Returns a clean single input and flat target array mapping
        return X_batch, y_batch

# --------------------------------------------------------------------------
# CHRONOLOGICAL CHOPPING AND PARALLEL STREAM GENERATION
# --------------------------------------------------------------------------
total_samples = len(df_1h)
train_end = int(total_samples * 0.70)
val_end = int(total_samples * 0.85)

# Use your target class column from Phase 6
raw_X = df_1h[feature_columns_list].to_numpy()
raw_y = df_1h['Target_Class'].to_numpy()

# Instantiate the clean baseline sequence streams
train_dataset = StandaloneSequenceStream(raw_X[:train_end], raw_y[:train_end], sequence_length=30, batch_size=64)
val_dataset = StandaloneSequenceStream(raw_X[train_end:val_end], raw_y[train_end:val_end], sequence_length=30, batch_size=64)
test_dataset = StandaloneSequenceStream(raw_X[val_end:], raw_y[val_end:], sequence_length=30, batch_size=64)

print("==========================================================")
print(f"✅ Phase 7 permanently updated. Data objects synced for standalone LSTM model.")

⚙️ PHASE 7: BUILDING CLEAN STANDALONE SEQUENCE GENERATOR
✅ Phase 7 permanently updated. Data objects synced for standalone LSTM model.


In [ ]:
# ==============================================================================
# 🥇 PHASE 8: LSTM FEATURE EXTRACTOR ARCHITECTURE
# Folder Mapping: src/models/lstm.py
# ==============================================================================
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, Dropout, BatchNormalization

print("==========================================================")
print("🧠 PHASE 8: CONSTRUCTING DEEP LSTM TEMPORAL LAYER NETWORK")
print("==========================================================")

def build_lstm_feature_extractor(sequence_length, feature_dimensions):
    """
    Constructs a production-grade multi-layer stacked LSTM network engineered
    specifically for CuDNN GPU processing acceleration.
    """
    # 1. Input Tensor Definition layer matching our pipeline dimensions exactly
    inputs = Input(shape=(sequence_length, feature_dimensions), name="Tensor_Sequence_Input")

    # 2. First Stacked LSTM Layer (Returns complete sequential tracking states)
    # Using default activation='tanh' and recurrent_activation='sigmoid' to force native CuDNN usage
    x = LSTM(
        units=64,
        return_sequences=True,
        kernel_regularizer=tf.keras.regularizers.l2(1e-4),
        name="CuDNN_LSTM_Layer_1"
    )(inputs)
    x = BatchNormalization(name="Batch_Norm_1")(x)
    x = Dropout(0.2, name="Temporal_Dropout_1")(x)

    # 3. Second Stacked LSTM Layer (Returns complete structural sequence dimension for Transformer attention mapping)
    x = LSTM(
        units=64,
        return_sequences=True,
        kernel_regularizer=tf.keras.regularizers.l2(1e-4),
        name="CuDNN_LSTM_Layer_2"
    )(x)
    x = BatchNormalization(name="Batch_Norm_2")(x)
    lstm_hidden_states = Dropout(0.2, name="Temporal_Dropout_2")(x)

    # 4. Initialize Functional API Model Container to decouple feature processing
    lstm_extractor_model = Model(inputs=inputs, outputs=lstm_hidden_states, name="LSTM_Feature_Extractor_Engine")

    print("\n==========================================================")
    print("📊 PHASE 8: LSTM COMPONENT ARCHITECTURE SCHEMATIC")
    print("==========================================================")
    lstm_extractor_model.summary()
    print("==========================================================")
    print("✅ Phase 8 LSTM feature extractor built and frozen for Transformer mapping.")

    return lstm_extractor_model

# Trigger Phase 8 Construction using parameters from Phase 7 pipeline
lstm_engine = build_lstm_feature_extractor(seq_len, num_features)

🧠 PHASE 8: CONSTRUCTING DEEP LSTM TEMPORAL LAYER NETWORK

📊 PHASE 8: LSTM COMPONENT ARCHITECTURE SCHEMATIC


Model: "LSTM_Feature_Extractor_Engine"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ Tensor_Sequence_Input           │ (None, 30, 38)         │             0 │
│ (InputLayer)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ CuDNN_LSTM_Layer_1 (LSTM)       │ (None, 30, 64)         │        26,368 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Batch_Norm_1                    │ (None, 30, 64)         │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Temporal_Dropout_1 (Dropout)    │ (None, 30, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ CuDNN_LSTM_Layer_2 (LSTM)       │ (None, 30, 64)         │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Batch_Norm_2                    │ (None, 30, 64)         │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Temporal_Dropout_2 (Dropout)    │ (None, 30, 64)         │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 59,904 (234.00 KB)

 Trainable params: 59,648 (233.00 KB)

 Non-trainable params: 256 (1.00 KB)

✅ Phase 8 LSTM feature extractor built and frozen for Transformer mapping.


In [ ]:
# ==============================================================================
# 🥇 PHASE 9: ADVANCED MULTI-TASK LSTM WITH CUSTOM 1:2 R:R CRITERIA COMPILATION
# Folder Mapping: src/models/lstm.py
# ==============================================================================
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, Dropout, GlobalAveragePooling1D

print("==========================================================")
print("🧠 PHASE 9: COMPILING CUSTOM 1:2 RISK:REWARD MULTI-TASK LSTM")
print("==========================================================")

# Custom Loss Framework directly implementing 1:2 Ratio constraints in Backprop
def custom_risk_reward_loss(y_true, y_pred):
    """
    Penalizes the regression variance heavily if the magnitude predicted
    does not satisfy or mathematically sustain the structural 1:2 R:R matrix.
    """
    # Absolute continuous target distance deviation
    base_mse = tf.math.square(y_true - y_pred)

    # Structural rule penalty: Check if magnitude satisfies minimum expected pip spread
    # dynamic scaling multiplier to ensure loss penalizes bad R:R entries
    rr_penalty = tf.where(tf.math.abs(y_pred) < (2.0 * tf.math.abs(y_true)), base_mse * 2.5, base_mse)
    return tf.math.reduce_mean(rr_penalty)

def build_architecture_aligned_rr_model(lstm_extractor, sequence_length, feature_dimensions):
    """
    Compiles the multi-task sequence architecture mapped tightly to
    your pipeline classification and regression output specifications.
    """
    inputs = Input(shape=(sequence_length, feature_dimensions), name="Global_Sequence_Input")

    # Slicing sequential features using your core baseline LSTM state
    lstm_features = lstm_extractor(inputs)
    pooled_features = GlobalAveragePooling1D(name="Shared_Temporal_Pooling")(lstm_features)

    # Latent Representation Space
    shared_dense = Dense(64, activation='relu', name="Shared_Feature_Layer")(pooled_features)
    shared_dense = Dropout(0.3, name="Shared_Dropout")(shared_dense)

    # Head A: Directional Target Classification
    class_dense = Dense(32, activation='relu', name="Class_Dense")(shared_dense)
    class_output = Dense(1, activation='sigmoid', name="Output_Classification")(class_dense)

    # Head B: Continuous Magnitude Target Regression
    reg_dense = Dense(32, activation='relu', name="Reg_Dense")(shared_dense)
    reg_output = Dense(1, activation='linear', name="Output_Regression")(reg_dense)

    # Complete Production Model Setup
    model = Model(inputs=inputs, outputs=[class_output, reg_output], name="Gold_RR_LSTM_System")

    # Compile with system standard rules + customized 1:2 ratio penalty constraints
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
        loss={
            "Output_Classification": tf.keras.losses.BinaryCrossentropy(),
            "Output_Regression": custom_risk_reward_loss  # Core math logic embedded here
        },
        loss_weights={
            "Output_Classification": 1.0,
            "Output_Regression": 0.5
        },
        metrics={
            "Output_Classification": [tf.keras.metrics.BinaryAccuracy(name='accuracy')],
            "Output_Regression": [tf.keras.metrics.MeanAbsoluteError(name='mae')]
        }
    )

    model.summary()
    return model

# Re-initialize the multi-task structural model matching your sequence configuration
# Note: Ensure you run this using the original 'MultiTaskSequenceStream' datasets
gold_rr_lstm_nn = build_architecture_aligned_rr_model(lstm_engine, 30, len(feature_columns_list))

🧠 PHASE 9: COMPILING CUSTOM 1:2 RISK:REWARD MULTI-TASK LSTM


Model: "Gold_RR_LSTM_System"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ Global_Sequence_In… │ (None, 30, 38)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ LSTM_Feature_Extra… │ (None, 30, 64)    │     59,904 │ Global_Sequence_… │
│ (Functional)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Shared_Temporal_Po… │ (None, 64)        │          0 │ LSTM_Feature_Ext… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Shared_Feature_Lay… │ (None, 64)        │      4,160 │ Shared_Temporal_… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Shared_Dropout      │ (None, 64)        │          0 │ Shared_Feature_L… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Class_Dense (Dense) │ (None, 32)        │      2,080 │ Shared_Dropout[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Reg_Dense (Dense)   │ (None, 32)        │      2,080 │ Shared_Dropout[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Output_Classificat… │ (None, 1)         │         33 │ Class_Dense[0][0] │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Output_Regression   │ (None, 1)         │         33 │ Reg_Dense[0][0]   │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 68,290 (266.76 KB)

 Trainable params: 68,034 (265.76 KB)

 Non-trainable params: 256 (1.00 KB)

In [ ]:
# ==============================================================================
# 🥇 PHASE 10: MULTI-TASK MODEL TRAINING ENGINE (STRICT 5 EPOCHS LOCKED)
# Folder Mapping: src/models/trainer.py
# ==============================================================================
print("==========================================================")
print("🚀 PHASE 10: EXECUTING MULTI-TASK LSTM TRAINING LOOP (5 EPOCHS)")
print("==========================================================")

def execute_multi_task_training(model, train_ds, val_ds):
    """
    Trains the multi-task model over exactly 5 epochs, feeding classification
    and regression logs cleanly into the gradient backprop pipeline.
    """
    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=20,       # Fixed limit maintained
        shuffle=False,   # Preserving timeline dependencies
        verbose=1
    )

    print("\n==========================================================")
    print("🎯 PHASE 10: MULTI-TASK PIPELINE SUCCESS")
    print("==========================================================")
    print("✅ Model optimized for Classification and Regression simultaneously over 5 epochs.")
    return history

# Run structural training pipeline execution
multitask_history = execute_multi_task_training(gold_multitask_nn, train_dataset, val_dataset)

🚀 PHASE 10: EXECUTING MULTI-TASK LSTM TRAINING LOOP (5 EPOCHS)
Epoch 1/20


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


1369/1369 ━━━━━━━━━━━━━━━━━━━━ 27s 17ms/step - Output_Classification_accuracy: 0.5260 - Output_Classification_loss: 0.6922 - Output_Regression_loss: 6.6765 - Output_Regression_mae: 1.5431 - loss: 4.0461 - val_Output_Classification_accuracy: 0.4944 - val_Output_Classification_loss: 0.7253 - val_Output_Regression_loss: 14.1442 - val_Output_Regression_mae: 2.4108 - val_loss: 7.8119
Epoch 2/20
1369/1369 ━━━━━━━━━━━━━━━━━━━━ 41s 17ms/step - Output_Classification_accuracy: 0.5319 - Output_Classification_loss: 0.6907 - Output_Regression_loss: 6.6500 - Output_Regression_mae: 1.5389 - loss: 4.0297 - val_Output_Classification_accuracy: 0.4945 - val_Output_Classification_loss: 0.7324 - val_Output_Regression_loss: 14.3078 - val_Output_Regression_mae: 2.4484 - val_loss: 7.8997
Epoch 3/20
1369/1369 ━━━━━━━━━━━━━━━━━━━━ 22s 16ms/step - Output_Classification_accuracy: 0.5347 - Output_Classification_loss: 0.6898 - Output_Regression_loss: 6.6210 - Output_Regression_mae: 1.5383 - loss: 4.0140 - val_Outpu

In [ ]:
# ==============================================================================
# 🥇 PHASE 11: MULTI-TASK EVALUATION ENGINE (OUT-OF-SAMPLE PERFORMANCE)
# Folder Mapping: src/models/evaluation.py
# ==============================================================================
import numpy as np
import tensorflow as tf

print("==========================================================")
print("📊 PHASE 11: RUNNING OUT-OF-SAMPLE MULTI-TASK EVALUATION")
print("==========================================================")

def execute_phase_11_multi_task_evaluation(model, val_end_idx, raw_X, raw_y_class, raw_y_reg, feature_cols):
    """
    Evaluates the dual-head model on the un-shuffled historical test slice,
    unifying classification metrics and continuous regression variance tracking.
    """
    # 1. Isolate test array data points matching Phase 7 partition rules
    test_X_raw = raw_X[val_end_idx:]
    test_y_class_raw = raw_y_class[val_end_idx:]
    test_y_reg_raw = raw_y_reg[val_end_idx:]

    # 2. Reconstruct direct sequence matrices for precise tracking metrics
    test_stream = MultiTaskSequenceStream(test_X_raw, test_y_class_raw, test_y_reg_raw, 30, 64)

    print("   • Running inference steps across multi-task operational layer...")
    evaluation_scores = model.evaluate(test_stream, verbose=0)

    # Keras multi-output structural list indexing maps to [total_loss, class_loss, reg_loss, class_acc, reg_mae]
    total_loss = evaluation_scores[0]
    class_acc = evaluation_scores[3] * 100.0
    reg_mae = evaluation_scores[4]

    print("\n==========================================================")
    print("🎯 PHASE 11: DUAL-HEAD TEST PERFORMANCE REPORT")
    print("==========================================================")
    print(f"  • Aggregated Pipeline Loss     : {total_loss:.4f}")
    print(f"  • Classification (Direction Acc): {class_acc:.2f}%")
    print(f"  • Regression (Magnitude MAE)   : {reg_mae:.6f}")
    print("==========================================================")
    print("✅ Phase 11 dual-head out-of-sample evaluation complete.")

    return evaluation_scores

# Trigger Phase 11 Performance Run using our active data boundaries
eval_metrics = execute_phase_11_multi_task_evaluation(
    gold_multitask_nn, val_end, raw_X, raw_y_class, raw_y_reg, feature_columns_list
)

📊 PHASE 11: RUNNING OUT-OF-SAMPLE MULTI-TASK EVALUATION
   • Running inference steps across multi-task operational layer...

🎯 PHASE 11: DUAL-HEAD TEST PERFORMANCE REPORT
  • Aggregated Pipeline Loss     : 35.0404
  • Classification (Direction Acc): 50.58%
  • Regression (Magnitude MAE)   : 3.795795
✅ Phase 11 dual-head out-of-sample evaluation complete.


In [ ]:
# ==============================================================================
# 🥇 PHASE 12: VECTORIZED BACKTESTING LAYER (STANDALONE LSTM SIMULATION)
# Folder Mapping: src/backtesting/vectorized.py
# ==============================================================================
import numpy as np
import pandas as pd

print("==========================================================")
print("📈 PHASE 12: INITIALIZING STANDALONE LSTM BACKTESTING ENGINE")
print("==========================================================")

def execute_phase_12_backtest(model, df_source, feature_cols, val_end_idx):
    """
    Simulates real market returns driven by standalone LSTM network predictions
    across the out-of-sample validation and testing partitions.
    """
    # 1. Slice out-of-sample data matching timeline matrices
    backtest_df = df_source.iloc[val_end_idx:].copy().reset_index(drop=True)

    # 2. Extract input sequence blocks for evaluation mapping
    X_raw = backtest_df[feature_cols].to_numpy(dtype=np.float32)

    # Preallocate batch blocks matching Lookback Context (30)
    sequence_length = 30
    indices = np.arange(sequence_length, len(backtest_df))

    X_sequences = np.empty((len(indices), sequence_length, len(feature_cols)), dtype=np.float32)
    for idx, end_point in enumerate(indices):
        X_sequences[idx] = X_raw[end_point - sequence_length : end_point]

    # 3. Model Prediction Layer Execution
    print("   • Generating model predictions for out-of-sample data...")
    predictions = model.predict(X_sequences, batch_size=64, verbose=0)

    # Align structural dataframe length to sequence offset boundaries
    aligned_df = backtest_df.iloc[sequence_length:].copy().reset_index(drop=True)

    # If using Multi-Task model, extract classification head results, else handle flat output
    if isinstance(predictions, list):
        aligned_df['Model_Signal_Prob'] = predictions[0].flatten()
    else:
        aligned_df['Model_Signal_Prob'] = predictions.flatten()

    # Binary conversion rule: Prob > 0.5 = Long (1), Else = Short (-1)
    aligned_df['Trading_Signal'] = np.where(aligned_df['Model_Signal_Prob'] > 0.5, 1, -1)

    # 4. Returns Simulation Modeling Matrix
    aligned_df['Market_Returns'] = aligned_df['Close'].pct_change().fillna(0.0)
    aligned_df['Strategy_Returns'] = aligned_df['Trading_Signal'].shift(1).fillna(0.0) * aligned_df['Market_Returns']

    # Compounding returns profiles
    aligned_df['Cum_Market_Returns'] = (1.0 + aligned_df['Market_Returns']).cumprod() - 1.0
    aligned_df['Cum_Strategy_Returns'] = (1.0 + aligned_df['Strategy_Returns']).cumprod() - 1.0

    print("\n==========================================================")
    print("📊 PHASE 12: ALGORITHMIC PERFORMANCE SIMULATION REPORT")
    print("==========================================================")
    print(f"  • Total Backtested Intervals  : {len(aligned_df)} Bars")
    print(f"  • Cumulative Benchmark Return : {aligned_df['Cum_Market_Returns'].iloc[-1]*100.0:.2f}%")
    print(f"  • Cumulative Strategy Return  : {aligned_df['Cum_Strategy_Returns'].iloc[-1]*100.0:.2f}%")
    print("==========================================================")
    print("✅ Phase 12 backtesting complete. Operational tracking active.")

    return aligned_df

# Trigger Phase 12 Execution across terminal dataset spaces
backtest_results_df = execute_phase_12_backtest(
    gold_multitask_nn, df_1h, feature_columns_list, val_end
)

📈 PHASE 12: INITIALIZING STANDALONE LSTM BACKTESTING ENGINE
   • Generating model predictions for out-of-sample data...

📊 PHASE 12: ALGORITHMIC PERFORMANCE SIMULATION REPORT
  • Total Backtested Intervals  : 18751 Bars
  • Cumulative Benchmark Return : 193.70%
  • Cumulative Strategy Return  : 13.08%
✅ Phase 12 backtesting complete. Operational tracking active.


In [ ]:
# ==============================================================================
# 🚀 OPTIMIZATION PHASE: HIGH-PERFORMANCE STANDALONE LSTM ARCHITECTURE
# Folder Mapping: src/models/lstm.py (Optimized Grid Setup)
# ==============================================================================
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, Dropout, BatchNormalization, GlobalAveragePooling1D

print("==========================================================")
print("🧠 INITIALIZING OPTIMIZED LSTM BASELINE ARCHITECTURE")
print("==========================================================")

def build_optimized_lstm_engine(sequence_length, feature_dimensions):
    """
    Constructs a regularized, high-capacity stacked LSTM framework
    tuned specifically for structural financial pattern recognition.
    """
    inputs = Input(shape=(sequence_length, feature_dimensions), name="Optimized_Input_Tensor")

    # Layer 1: Increased unit capacity to capture fine-grained market dynamics
    x = LSTM(
        units=128,
        return_sequences=True,
        kernel_regularizer=tf.keras.regularizers.l2(5e-4),
        name="Optimized_LSTM_1"
    )(inputs)
    x = BatchNormalization(name="Norm_Layer_1")(x)
    x = Dropout(0.3, name="Dropout_Layer_1")(x)

    # Layer 2: Deep feature aggregation abstraction
    x = LSTM(
        units=64,
        return_sequences=True,
        kernel_regularizer=tf.keras.regularizers.l2(5e-4),
        name="Optimized_LSTM_2"
    )(x)
    x = BatchNormalization(name="Norm_Layer_2")(x)
    x = Dropout(0.3, name="Dropout_Layer_2")(x)

    # Dimensionality Reduction & Dense Classification Head
    pooled = GlobalAveragePooling1D(name="State_Pooling_Head")(x)
    dense_projection = Dense(32, activation='relu', name="Dense_Compressor")(pooled)
    final_dropout = Dropout(0.2, name="Final_Regularization")(dense_projection)

    # Multi-task conceptual output (Flat binary classification signature for direct strategy mapping)
    outputs = Dense(1, activation='sigmoid', name="Directional_Probability")(final_dropout)

    # Formulate and compile the optimized standalone network graph
    optimized_model = Model(inputs=inputs, outputs=outputs, name="Supercharged_LSTM_Engine")

    # Utilizing an aggressive optimizer schedule framework via customized learning rate parameters
    optimized_model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=5e-4),
        loss=tf.keras.losses.BinaryCrossentropy(),
        metrics=[
            tf.keras.metrics.BinaryAccuracy(name='accuracy'),
            tf.keras.metrics.Precision(name='precision'),
            tf.keras.metrics.Recall(name='recall')
        ]
    )

    optimized_model.summary()
    return optimized_model

# Re-instantiate the model engine using existing shape metadata
optimized_lstm_nn = build_optimized_lstm_engine(seq_len, num_features)

🧠 INITIALIZING OPTIMIZED LSTM BASELINE ARCHITECTURE


Model: "Supercharged_LSTM_Engine"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ Optimized_Input_Tensor          │ (None, 30, 38)         │             0 │
│ (InputLayer)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Optimized_LSTM_1 (LSTM)         │ (None, 30, 128)        │        85,504 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Norm_Layer_1                    │ (None, 30, 128)        │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Dropout_Layer_1 (Dropout)       │ (None, 30, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Optimized_LSTM_2 (LSTM)         │ (None, 30, 64)         │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Norm_Layer_2                    │ (None, 30, 64)         │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Dropout_Layer_2 (Dropout)       │ (None, 30, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ State_Pooling_Head              │ (None, 64)             │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Dense_Compressor (Dense)        │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Final_Regularization (Dropout)  │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Directional_Probability (Dense) │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 137,793 (538.25 KB)

 Trainable params: 137,409 (536.75 KB)

 Non-trainable params: 384 (1.50 KB)

In [ ]:
# ==============================================================================
# 🚀 OPTIMIZATION PHASE: HIGH-PERFORMANCE STANDALONE LSTM ARCHITECTURE
# Folder Mapping: src/models/lstm.py (Optimized Grid Setup)
# ==============================================================================
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, Dropout, BatchNormalization, GlobalAveragePooling1D

print("==========================================================")
print("🧠 INITIALIZING OPTIMIZED LSTM BASELINE ARCHITECTURE")
print("==========================================================")

def build_optimized_lstm_engine(sequence_length, feature_dimensions):
    """
    Constructs a regularized, high-capacity stacked LSTM framework
    tuned specifically for structural financial pattern recognition.
    """
    inputs = Input(shape=(sequence_length, feature_dimensions), name="Optimized_Input_Tensor")

    # Layer 1: Increased unit capacity to capture fine-grained market dynamics
    x = LSTM(
        units=128,
        return_sequences=True,
        kernel_regularizer=tf.keras.regularizers.l2(5e-4),
        name="Optimized_LSTM_1"
    )(inputs)
    x = BatchNormalization(name="Norm_Layer_1")(x)
    x = Dropout(0.3, name="Dropout_Layer_1")(x)

    # Layer 2: Deep feature aggregation abstraction
    x = LSTM(
        units=64,
        return_sequences=True,
        kernel_regularizer=tf.keras.regularizers.l2(5e-4),
        name="Optimized_LSTM_2"
    )(x)
    x = BatchNormalization(name="Norm_Layer_2")(x)
    x = Dropout(0.3, name="Dropout_Layer_2")(x)

    # Dimensionality Reduction & Dense Classification Head
    pooled = GlobalAveragePooling1D(name="State_Pooling_Head")(x)
    dense_projection = Dense(32, activation='relu', name="Dense_Compressor")(pooled)
    final_dropout = Dropout(0.2, name="Final_Regularization")(dense_projection)

    # Multi-task conceptual output (Flat binary classification signature for direct strategy mapping)
    outputs = Dense(1, activation='sigmoid', name="Directional_Probability")(final_dropout)

    # Formulate and compile the optimized standalone network graph
    optimized_model = Model(inputs=inputs, outputs=outputs, name="Supercharged_LSTM_Engine")

    # Utilizing an aggressive optimizer schedule framework via customized learning rate parameters
    optimized_model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=5e-4),
        loss=tf.keras.losses.BinaryCrossentropy(),
        metrics=[
            tf.keras.metrics.BinaryAccuracy(name='accuracy'),
            tf.keras.metrics.Precision(name='precision'),
            tf.keras.metrics.Recall(name='recall')
        ]
    )

    optimized_model.summary()
    return optimized_model

# Re-instantiate the model engine using existing shape metadata
optimized_lstm_nn = build_optimized_lstm_engine(seq_len, num_features)

🧠 INITIALIZING OPTIMIZED LSTM BASELINE ARCHITECTURE


Model: "Supercharged_LSTM_Engine"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ Optimized_Input_Tensor          │ (None, 30, 38)         │             0 │
│ (InputLayer)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Optimized_LSTM_1 (LSTM)         │ (None, 30, 128)        │        85,504 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Norm_Layer_1                    │ (None, 30, 128)        │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Dropout_Layer_1 (Dropout)       │ (None, 30, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Optimized_LSTM_2 (LSTM)         │ (None, 30, 64)         │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Norm_Layer_2                    │ (None, 30, 64)         │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Dropout_Layer_2 (Dropout)       │ (None, 30, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ State_Pooling_Head              │ (None, 64)             │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Dense_Compressor (Dense)        │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Final_Regularization (Dropout)  │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Directional_Probability (Dense) │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 137,793 (538.25 KB)

 Trainable params: 137,409 (536.75 KB)

 Non-trainable params: 384 (1.50 KB)

In [ ]:
# ==============================================================================
# 📊 MODEL PREDICTION CHECKER & VERIFICATION ENGINE
# ==============================================================================
import numpy as np
import pandas as pd

print("==========================================================")
print("🧐 RUNNING REAL-TIME MODEL PREDICTION VERIFICATION")
print("==========================================================")

# 1. Fetch a batch from the test dataset
X_test_batch, y_true_batch = test_dataset[0]

# 2. Generate predictions from your optimized model
y_pred_probs = optimized_lstm_nn.predict(X_test_batch, verbose=0).flatten()
# Convert probabilities to crisp binary choices (0 or 1)
y_pred_binary = (y_pred_probs > 0.5).astype(int)

# 3. Create a comparison dataframe for visual inspection
comparison_df = pd.DataFrame({
    'Actual_Target': y_true_batch,
    'Predicted_Probability': y_pred_probs,
    'Model_Prediction': y_pred_binary,
    'Is_Correct': y_true_batch == y_pred_binary
})

print("\n📊 FIRST 10 TEST PREDICTIONS COMPARISON:")
print("==========================================================")
print(comparison_df.head(10).to_string(index=False))
print("==========================================================")

# 4. Calculate batch accuracy directly
correct_count = comparison_df['Is_Correct'].sum()
total_count = len(comparison_df)
batch_acc = (correct_count / total_count) * 100.0

print(f"🎯 Batch Verification Accuracy: {batch_acc:.2f}% ({correct_count}/{total_count} Match)")
print("==========================================================")

🧐 RUNNING REAL-TIME MODEL PREDICTION VERIFICATION

📊 FIRST 10 TEST PREDICTIONS COMPARISON:
 Actual_Target  Predicted_Probability  Model_Prediction  Is_Correct
             1               0.033774                 0       False
             1               0.033474                 0       False
             1               0.030896                 0       False
             0               0.028291                 0        True
             0               0.026211                 0        True
             1               0.025635                 0       False
             1               0.026388                 0       False
             0               0.027354                 0        True
             0               0.028539                 0        True
             1               0.030072                 0       False
🎯 Batch Verification Accuracy: 53.12% (34/64 Match)
